# Notebook 3: Interventions and Qualitative Analysis

Colab-ready workflow for cloning the GitHub repo, mounting Google Drive, reusing stored checkpoints and candidate-circuit artifacts, and running held-out residual-patch interventions with persistent outputs.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/jacobposchl/model-interpretability.git'
REPO_DIR = Path('/content/model_interpretability' if IN_COLAB else Path.cwd()).resolve()

if IN_COLAB:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)

    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/flow_circuits')
    DATA_ROOT = Path('/content/drive/MyDrive/ctls/data')
else:
    if not (REPO_DIR / 'flow_circuits').exists():
        raise RuntimeError('Run this notebook from the repository root or in Google Colab.')
    os.chdir(REPO_DIR)
    DRIVE_ROOT = REPO_DIR / 'artifacts' / 'local_notebook_runtime'
    DATA_ROOT = DRIVE_ROOT / 'data'

EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments'
NOTEBOOK_ROOT = DRIVE_ROOT / 'notebook_runs'
for path in (DRIVE_ROOT, DATA_ROOT, EXPERIMENT_ROOT, NOTEBOOK_ROOT):
    path.mkdir(parents=True, exist_ok=True)

CONFIG_ROOT = REPO_DIR / 'configs' / 'flow'
print({'repo_dir': str(REPO_DIR), 'drive_root': str(DRIVE_ROOT), 'data_root': str(DATA_ROOT), 'experiment_root': str(EXPERIMENT_ROOT)})


In [ ]:
from flow_circuits.data import build_cifar10_splits
from flow_circuits.interventions import run_circuit_interventions
from flow_circuits.training import collect_model_outputs, load_components_from_checkpoint


In [ ]:
CONFIG_PATH = str(CONFIG_ROOT / 'resnet18_base.yaml')
BACKBONE_WEIGHTS_PATH = None
CHECKPOINT_PATH = None
CIRCUITS_PATH = None
OUTPUT_DIR = str(NOTEBOOK_ROOT / 'nb03')
QUICK_MODE = True


In [ ]:
import copy
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLI_MODULES = {
    'flow-train': 'flow_circuits.cli.train',
    'flow-discover': 'flow_circuits.cli.discover',
    'flow-intervene': 'flow_circuits.cli.intervene',
}

def run_flow_cli(command: str, *args: str) -> None:
    def _stream(cmd):
        import os
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            cwd=REPO_DIR,
            env=env,
        )
        while True:
            line = process.stdout.readline()
            if not line:
                break
            print(line, end="", flush=True)
        process.wait()
        if process.returncode != 0:
            raise subprocess.CalledProcessError(process.returncode, cmd)

    try:
        _stream([command, *args])
    except FileNotFoundError:
        _stream([sys.executable, '-m', CLI_MODULES[command], *args])

with open(CONFIG_PATH, 'r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)
config['data']['data_dir'] = str(DATA_ROOT)
config['data']['download'] = False
if BACKBONE_WEIGHTS_PATH:
    config['backbone']['weights_path'] = str(BACKBONE_WEIGHTS_PATH)
config['logging']['checkpoint_dir'] = str(EXPERIMENT_ROOT / config['experiment']['name'])

effective_config = copy.deepcopy(config)
if QUICK_MODE:
    effective_config['data']['batch_size'] = 32
    effective_config['data']['num_workers'] = 0
    effective_config['training']['phase_epochs'] = {'phase_a': 1, 'phase_b': 1, 'phase_c': 0}
    effective_config['training']['baseline_fit_images'] = 128
    effective_config['training']['baseline_eval_images'] = 128
    effective_config['training']['validation_images'] = 64
    effective_config['training']['alignment_max_pairs'] = 256
    effective_config['discovery']['max_images'] = 256
    effective_config['discovery']['bootstrap_iterations'] = 4
    effective_config['interventions']['max_images'] = 128
    effective_config['logging']['checkpoint_dir'] = str(OUTPUT_DIR / 'quick_checkpoints')

EFFECTIVE_CONFIG = OUTPUT_DIR / ('quick_config.yaml' if QUICK_MODE else 'effective_config.yaml')
EFFECTIVE_CONFIG.write_text(yaml.safe_dump(effective_config, sort_keys=False), encoding='utf-8')

EFFECTIVE_CHECKPOINT = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else Path(effective_config['logging']['checkpoint_dir']) / 'final.pt'
EFFECTIVE_CIRCUITS = Path(CIRCUITS_PATH) if CIRCUITS_PATH else Path(effective_config['logging']['checkpoint_dir']) / 'candidate_circuits.json'
SUMMARY_JSON = OUTPUT_DIR / 'intervention_summary.json'
SUMMARY_CSV = SUMMARY_JSON.with_suffix('.csv')
if (not CHECKPOINT_PATH) and effective_config['backbone'].get('require_trained_checkpoint', False) and not effective_config['backbone'].get('weights_path'):
    raise RuntimeError(
        'Set BACKBONE_WEIGHTS_PATH to your trained CIFAR-10 ResNet checkpoint before training. '
        'The canonical configs now require a supervised backbone checkpoint for scientifically valid logits.'
    )
print({'config': str(EFFECTIVE_CONFIG), 'checkpoint': str(EFFECTIVE_CHECKPOINT), 'circuits': str(EFFECTIVE_CIRCUITS), 'summary': str(SUMMARY_JSON)})


In [ ]:
if not EFFECTIVE_CHECKPOINT.exists():
    run_flow_cli('flow-train', '--config', str(EFFECTIVE_CONFIG))

if not EFFECTIVE_CIRCUITS.exists():
    run_flow_cli('flow-discover', '--checkpoint', str(EFFECTIVE_CHECKPOINT), '--output', str(EFFECTIVE_CIRCUITS))

if not SUMMARY_JSON.exists():
    run_flow_cli('flow-intervene', '--checkpoint', str(EFFECTIVE_CHECKPOINT), '--circuits', str(EFFECTIVE_CIRCUITS), '--output', str(SUMMARY_JSON))

intervention_summary = json.loads(SUMMARY_JSON.read_text(encoding='utf-8'))
circuits_artifact = json.loads(EFFECTIVE_CIRCUITS.read_text(encoding='utf-8'))
{'n_intervention_results': len(intervention_summary), 'n_validated': sum(1 for row in intervention_summary if row.get('validated'))}


In [ ]:
preview = intervention_summary[:5]
preview


In [ ]:
if intervention_summary:
    labels = [f"C{row['circuit_id']}" for row in intervention_summary]
    member = [row['mean_member_delta_margin'] for row in intervention_summary]
    nonmember = [row['mean_nonmember_delta_margin'] for row in intervention_summary]
    random_node = [row['mean_random_node_delta_margin'] for row in intervention_summary]
    random_cell = [row['mean_random_cell_delta_margin'] for row in intervention_summary]

    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(max(6, 1.2 * len(labels)), 4))
    ax.plot(x, member, marker='o', label='members')
    ax.plot(x, nonmember, marker='o', label='matched non-members')
    ax.plot(x, random_node, marker='o', label='random-node control')
    ax.plot(x, random_cell, marker='o', label='random-cell control')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('Mean delta margin')
    ax.set_title('Held-out intervention effect summary')
    ax.legend()
    plt.tight_layout()
else:
    print('No intervention rows available to visualize.')


In [ ]:
max_plots = min(4, len(circuits_artifact['circuits']))
grid_size = circuits_artifact['metadata']['grid_size']
if max_plots == 0:
    print('No candidate circuits available in the artifact.')
else:
    fig, axes = plt.subplots(1, max_plots, figsize=(4 * max_plots, 3.5))
    if max_plots == 1:
        axes = [axes]
    for axis, circuit in zip(axes, circuits_artifact['circuits'][:max_plots]):
        heatmap = np.zeros((grid_size, grid_size), dtype=float)
        for _, cell_idx in circuit['active_nodes']:
            row = cell_idx // grid_size
            col = cell_idx % grid_size
            heatmap[row, col] += 1.0
        image = axis.imshow(heatmap, cmap='viridis')
        axis.set_title(f"Circuit {circuit['id']} footprint")
        plt.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    plt.tight_layout()


In [ ]:
validated = [row for row in intervention_summary if row.get('validated')]
if validated:
    validated
else:
    print('No circuits passed the current intervention validation rule for this summary.')
